# Significance Testing via Permutation Tests

- Text level was shuffled in training w/ a different random seed for each permutation

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.utils import ALPHA_LIST
import pandas as pd
import numpy as np
import shlex
import json
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from src.perm_res import (
    get_p_df,
    expand_alpha_rows,
    collapse_hypothesis,
    get_collapsed_null,
    get_ranking,
    get_avg_r2,
)

# Part 1- Get best 3Q score

## Explore r2

In [ ]:
parent_base_df = pd.read_parquet(
    BASE_PATH / "perm_res/preds/baseline/parent/pred_df.parquet"
)
self_base_df = pd.read_parquet(
    BASE_PATH / "perm_res/preds/baseline/self/pred_df.parquet"
)

In [ ]:
def compare_r2s(df):
	r2_df = get_avg_r2(df)

	r2s_3 = r2_df[r2_df["num_layers"] == 3]
	best_3Q_row = r2s_3.sort_values(by="r2_Q", ascending=False).iloc[0]
	best_alpha = best_3Q_row["alpha"].split("_")[0]
	best_3Q_val = best_3Q_row["r2_Q"]
	r2s_3['best_Q'] = best_3Q_val
	r2s_3["diff"] = r2s_3["r2_T"] - best_3Q_val

	r2s_2 = r2_df[r2_df["num_layers"] == 2]
	r2s_2["diff"] = r2s_2["r2_T"] - r2s_2["r2_S"]
	return (
		r2s_2.sort_values(by="diff", ascending=False),
		r2s_3.sort_values(by="diff", ascending=False),
		best_alpha,
	)

Parent

In [ ]:
r2s_2_parent, r2s_3_parent, best_alpha_parent = compare_r2s(parent_base_df)
r2s_2_self, r2s_3_self, best_alpha_self = compare_r2s(self_base_df)

In [ ]:
display(r2s_3_parent)
display(r2s_2_parent)

In [ ]:
display(r2s_3_self)
display(r2s_2_self)

## Aggregate r^2 values across perms

- One table per impact/num-layers combination
- Each table contains r2_S,r2_Q,r2_T for some model configuration

In [ ]:
def create_cmd(num_layers, import_dir, export_path, n_threads):
    cmd_str_list = [
        f"export PYTHONPATH={BASE_PATH};",
        f"export OMP_NUM_THREADS={n_threads};",
        "uv run",
        "python -m src.perm_res",
        "--num_layers",
        str(num_layers),
        "--import_dir",
        str(import_dir),
        "--export_path",
        str(export_path),
    ]
    return " ".join(cmd_str_list)

In [ ]:
perm_check_start = 0
perm_check_end = 1000  # not inclusive
INCLUDE_BASELINE = True

In [ ]:
full_cmd_list = []
loop_dir = BASE_PATH / "perm_res" / "preds"
swarm_dir = BASE_PATH / "swarm_dir/perm_r2"
cmd_dir = swarm_dir / "commands"
log_dir = swarm_dir / "logs"
r2_dir = BASE_PATH / "perm_res" / "r2_tables"
for impact_type in ["parent", "self"]:
    for num_layers in [2, 3]:
        swarm_path = cmd_dir / f"{impact_type}_{num_layers}.swarm"
        swarm_path.parent.mkdir(exist_ok=True, parents=True)
        if swarm_path.exists():
            swarm_path.unlink()
        cmd_list = []
        for perm_dir in loop_dir.iterdir():
            export_path = r2_dir / perm_dir.name / f"{impact_type}_{num_layers}.parquet"
            perm_name = perm_dir.name
            if perm_name == "baseline":
                if not INCLUDE_BASELINE:
                    continue
            else:
                perm_num = int(perm_name.split("_")[1].strip(""))
                if (perm_num < perm_check_start) or (perm_num >= perm_check_end):
                    continue
            print(perm_name)

            cmd_str = create_cmd(
                num_layers=num_layers,
                import_dir=perm_dir / impact_type,
                n_threads=2,
                export_path=export_path,
            )
            cmd_list.append(cmd_str)
        swarm_path.write_text("\n".join(cmd_list))
        full_cmd_list += cmd_list
print(f"Num commands: {len(full_cmd_list)}")

Run w/ swarm

In [ ]:
from src.swarm import run_perm_r2_swarm

run_perm_r2_swarm(
    impact_list=["parent", "self"],
    layer_list=[2, 3],
    log_dir=log_dir,
    cmd_dir=cmd_dir,
    n_batch_for_3=6,
)

Run sequentially

In [ ]:
# import subprocess

# for iteration, cmd in enumerate(full_cmd_list):
#     print(f"{iteration +1}/{len(full_cmd_list)}...")
#     print(cmd)
#     subprocess.run(cmd, shell=True, check=True)

## Get p-values

### Raw p-vals for each model config given observed/permutted diffs
- 2 layer: diff = r2_T - r2_S
- 3 layer: diff = r2_T - r2_Q
- NULL distribvution = diff of max(2T) - max(2S/3Q) of each permutation
    - only within hypotheses

In [ ]:
base_imp_path = BASE_PATH / "perm_res" / "r2_tables"
# Dict of the form {perm_name: df} for each of the 4 types of dfs (parent_2, self_2, parent_3, self_3)
parent_2_df_dict = {
    perm_dir.name: pd.read_parquet(base_imp_path / perm_dir.name / "parent_2.parquet")
    for perm_dir in base_imp_path.iterdir()
}
self_2_df_dict = {
    perm_dir.name: pd.read_parquet(base_imp_path / perm_dir.name / "self_2.parquet")
    for perm_dir in base_imp_path.iterdir()
}
parent_3_df_dict = {
    perm_dir.name: pd.read_parquet(base_imp_path / perm_dir.name / "parent_3.parquet")
    for perm_dir in base_imp_path.iterdir()
}
self_3_df_dict = {
    perm_dir.name: pd.read_parquet(base_imp_path / perm_dir.name / "self_3.parquet")
    for perm_dir in base_imp_path.iterdir()
}

In [ ]:
p_parent_2, null_parent_2 = get_p_df(num_layers=2, r2_df_dict=parent_2_df_dict)
p_parent_3, null_parent_3 = get_p_df(
    num_layers=3,
    r2_df_dict=parent_3_df_dict,
    best_alpha=best_alpha_parent,
)
p_self_2, null_self_2 = get_p_df(num_layers=2, r2_df_dict=self_2_df_dict)
p_self_3, null_self_3 = get_p_df(
    num_layers=3, r2_df_dict=self_3_df_dict, best_alpha=best_alpha_self
)

### Conjuction p-vals

1) Single diff for each model (minimum across hypotheses)
    - Will have to duplicate 2-layer model configs to match 3-model configs

2) Null distribution of diffs (one per permutation)
    - Start w/ 3-dimensional data
        - Axis1: permutations
        - Axis2: model config
        - Axis3: Hypotheses

    - Collapse axis3 by taking minimum diff across all hypotheses
    - Collapse axis2 by taking maximum diff across all model configs
    - End with list of 1001 diffs (including baseline)

In [ ]:
def collapse_hypothesis(
    df1, df2, col1, col2, remove_p_col=True, remove_baseline_diffs=True
):
    """
    Collapses hypothesis across 2 and 3 layers by taking the min of the baseline diffs.
    Returns a df with columns: agg, llama_model, desc_seg, alpha, min_diff
    """
    combined = pd.merge(
        df1,
        df2,
        on=["agg_type", "llama_model", "desc_seg", "alpha"],
        suffixes=("_2", "_3"),
    )
    if remove_p_col:
        combined = combined.drop(columns=["p_val_2", "p_val_3"])
    combined["min_diff"] = combined.apply(
        lambda row: min(getattr(row, col1), getattr(row, col2)), axis=1
    )
    if remove_baseline_diffs:
        combined = combined.drop(columns=["baseline_diff_2", "baseline_diff_3"])
    return combined.sort_values("min_diff", ascending=False)


def get_collapsed_null(
    impact_type, best_alpha, perm_dir_base=BASE_PATH / "perm_res" / "r2_tables"
):
    """
    Creates null distribution by first collapsing hypothesis axis (min) then collapsing model axis (max)
    for each permutation. Results in a single diff value per permutation to create a shared null distribution
    for a given impact_type.

    Params
    -----
    impact_type: str
        One of ['self', 'parent']

    Returns
    -------
    null_dist_hyp_1: list[float]
        List of max( <-- across all models
                    min( <-- for each model
                        (r2_2T - r2_2S),
                        (r2_3T-max(r2_3Q))
                    )
                )
    null_dist_hyp_2: list[float]
        List of max( <-- across all models
                    (r2_2T - max(r2_3Q))
                )
    baseline_df: pandas DF

    """
    null_dist_hyp_1 = []
    null_dist_hyp_2 = []
    baseline_df = None
    for perm_dir in perm_dir_base.iterdir():
        ## 3-layer r2 prep
        r2_df_3 = pd.read_parquet(perm_dir / f"{impact_type}_3.parquet")
        # Ensure only using best 3Q alpha
        r2_df_3 = r2_df_3[r2_df_3["alpha"].str.split("_").str[0] == best_alpha]
        assert np.allclose(r2_df_3["r2_Q"].values, r2_df_3["r2_Q"].values[0]), r2_df_3[
            "r2_Q"
        ].unique()
        r2_df_3["diff_3T_3Q"] = r2_df_3["r2_T"] - r2_df_3["r2_Q"]

        ## 2-layer r2 prep
        r2_df_2 = pd.read_parquet(perm_dir / f"{impact_type}_2.parquet")
        r2_df_2_dup = r2_df_2.copy()
        r2_df_2_dup["alpha"] = f"{best_alpha}_" + r2_df_2["alpha"]
        r2_df_2_dup["diff_2T_2S"] = r2_df_2_dup["r2_T"] - r2_df_2_dup["r2_S"]

        ## Collapse hypothesis axis
        combined_df = collapse_hypothesis(
            r2_df_2_dup,
            r2_df_3,
            "diff_3T_3Q",
            "diff_2T_2S",
            remove_p_col=False,
            remove_baseline_diffs=False,
        )
        combined_df = combined_df.drop(["num_layers_2", "num_layers_3"], axis=1)
        ## Get hypothesis 2 vals
        assert combined_df["r2_Q_3"].nunique() == 1
        combined_df["diff_2T_3Q"] = combined_df["r2_T_2"] - combined_df["r2_Q_3"]
        # Save baseline df for models
        if perm_dir.name == "baseline":
            baseline_df = combined_df
            # Mark min_diff NA when either r^2 T-layer <0
            baseline_df["min_diff"] = np.where(
                (baseline_df["r2_T_2"] < 0) | (baseline_df["r2_T_3"] < 0),
                np.nan,
                baseline_df["min_diff"],
            )
            baseline_df["diff_2T_3Q"] = np.where(
                (baseline_df["r2_T_2"] < 0),
                np.nan,
                baseline_df["diff_2T_3Q"],
            )
        max_diff_hyp_1 = combined_df["min_diff"].max()
        null_dist_hyp_1.append(max_diff_hyp_1)
        max_diff_hyp_2 = combined_df["diff_2T_3Q"].max()
        null_dist_hyp_2.append(max_diff_hyp_2)
    return null_dist_hyp_1, null_dist_hyp_2, baseline_df

In [ ]:
null_dist_parent_1, null_dist_parent_2, parent_baseline = get_collapsed_null(
    impact_type="parent", best_alpha=best_alpha_parent
)
null_dist_self_1, null_dist_self_2, self_baseline = get_collapsed_null(
    impact_type="self", best_alpha=best_alpha_self
)

In [ ]:
def get_p_vals(
    baseline_df,
    null_dist,
    title,
    diff_col="min_diff",
    show_plot=True,
    xlim=(-0.05, 0.1),
    ylim=(0, 50),
):
    baseline_df["p_val"] = baseline_df[diff_col].apply(
        lambda baseline: get_ranking(baseline, null_dist)
    )
    baseline_df["p_val"] = np.where(
        baseline_df[diff_col].isna(), np.nan, baseline_df["p_val"]
    )
    combined_sig = baseline_df[(baseline_df["p_val"] < 0.05)].sort_values(
        diff_col, ascending=False
    )
    ## Includes NA min diffs (at least one of r2_T_2 and/or r2_T_3 <0)
    non_sig = baseline_df[~(baseline_df["p_val"] < 0.05)].sort_values(
        diff_col, ascending=False
    )
    assert len(combined_sig) + len(non_sig) == len(baseline_df)
    print(f"Significant model configs: {len(combined_sig)} / {len(baseline_df)}")
    if show_plot:
        plt.figure(figsize=(10, 5))
        plt.xlim(xlim)
        plt.ylim(ylim)
        if len(combined_sig) > 0:
            plt.hist(
                combined_sig[diff_col],
                bins=25,
                alpha=0.6,
                color="green",
                label=f"SIG Observed diffs ({len(combined_sig)} / {len(baseline_df)})",
            )
        plt.hist(
            non_sig[diff_col],
            bins=50,
            alpha=0.6,
            color="red",
            label=f"NON-SIG Observed diffs ({len(non_sig)} / {len(baseline_df)})",
        )
        plt.hist(
            null_dist,
            bins=50,
            alpha=0.6,
            color="grey",
            label="Null distribution (1000 perms)",
        )
        plt.title(title)
        plt.xlabel("Difference in R²")
        plt.ylabel("Frequency")
        plt.legend()
        plt.show()
    return combined_sig, non_sig

Hypotheses 1/2 comparing 2S/2T + 3Q/3T

In [ ]:
sig_parent_1, non_sig_parent_1 = get_p_vals(
    parent_baseline,
    null_dist_parent_1,
    title="Null and observed distribution for PARENT conjunction hypotheses",
    diff_col="min_diff",
    show_plot=True,
    xlim=(-0.08, 0.08),
    ylim=(0, 300),
)
sig_self_2, non_sig_self_2 = get_p_vals(
    self_baseline,
    null_dist_self_1,
    title="Null and observed distribution for SELF conjunction hypotheses",
    diff_col="min_diff",
    show_plot=True,
    xlim=(-0.06, 0.06),
    ylim=(0, 215),
)

Hypothesis 3 comparing 3Q/2T 
- inherently flawed bc this only uses data where 2T is permuted. 
- Answers question of signal(2T) > 0 compared to 3Q baseline, 
    - NOT signal(2T) == signal(3Q) questionZ

In [ ]:
sig_parent_2, non_sig_parent_2 = get_p_vals(
    parent_baseline,
    null_dist_parent_2,
    diff_col="diff_2T_3Q",
    title="Null and observed distribution for PARENT naive 2T/3Q hypothesis",
    xlim=(-0.1, 0.02),
    ylim=(0, 200),
)
sig_self_2, non_sig_self_2 = get_p_vals(
    self_baseline,
    null_dist_self_2,
    diff_col="diff_2T_3Q",
    title="Null and observed distribution for SELF naive 2T/3Q hypothesis",
    xlim=(-0.1, 0.04),
    ylim=(0, 250),
)

## Hypothesis 3--> 2T-3Q>0

In [ ]:
from shutil import rmtree

r2_export_dir = BASE_PATH / "perm_res/switch/r2_dfs"
perm_ids = [f"{i:03d}" for i in range(1, 1000)]

### Create r2 tables --> only run once

In [ ]:
parent_permed = pd.read_parquet(BASE_PATH / "perm_res/switch/perm_diffs/parent.parquet")
self_permed = pd.read_parquet(BASE_PATH / "perm_res/switch/perm_diffs/self.parquet")

In [ ]:
def compute_r2s_switch(g):
    scores = {
        "r2_2T": r2_score(g["true"], g["pred_2T"]),
        "r2_3Q": r2_score(g["true"], g["pred_3Q"]),
    }
    for pid in perm_ids:
        scores[f"r2_2T_perm_{pid}"] = r2_score(g["true"], g[f"pred_2T_perm_{pid}"])
        scores[f"r2_3Q_perm_{pid}"] = r2_score(g["true"], g[f"pred_3Q_perm_{pid}"])
    return pd.Series(scores)


def get_r2_df(perm_df):
    return (
        perm_df.groupby(["agg_type", "llama_model", "desc_seg", "alpha_2T", "repeat"])
        # compute r^2 for each perm (including baseline) within a repeat (across splits) for each model
        .apply(compute_r2s_switch)
        .reset_index()
        # take avg r^2 across repeats
        .groupby(["agg_type", "llama_model", "desc_seg", "alpha_2T"])[
            ["r2_2T", "r2_3Q"]
            + [f"r2_2T_perm_{pid}" for pid in perm_ids]
            + [f"r2_3Q_perm_{pid}" for pid in perm_ids]
        ]
        .mean()
        .reset_index()
    )

In [ ]:
parent_r2_df = get_r2_df(parent_permed)
self_r2_df = get_r2_df(self_permed)

In [ ]:
if r2_export_dir.exists():
    rmtree(r2_export_dir)
r2_export_dir.mkdir(exist_ok=False, parents=True)
parent_r2_df.to_parquet(r2_export_dir / "parent.parquet")
self_r2_df.to_parquet(r2_export_dir / "self.parquet")

### Analyze tables

In [ ]:
parent_r2_df_switch = pd.read_parquet(r2_export_dir / "parent.parquet")
self_r2_df_switch = pd.read_parquet(r2_export_dir / "self.parquet")

In [ ]:
def get_null_dist_switch(r2_df):
    """
    Instead of taking the objective max diff for null distribution,
        take the diff of the max 2T r^2 value and max 3Q r^2 in a given permutation

    Otherwise, diff vals are inflated by really poor (negative) 3Q r^2
    """
    df_sub = r2_df[["agg_type", "llama_model", "desc_seg", "alpha_2T"]].copy()
    null_diffs = []
    assert r2_df["r2_3Q"].nunique() == 1
    for perm_id in ["baseline"] + perm_ids:
        suffix = "" if perm_id == "baseline" else f"_perm_{perm_id}"
        best_r2_2T = r2_df[f"r2_2T{suffix}"].max()
        best_r2_3Q = r2_df[f"r2_3Q{suffix}"].max()
        max_diff = best_r2_2T - best_r2_3Q
        if perm_id == "baseline":
            df_sub["diff_baseline"] = r2_df["r2_2T"] - r2_df["r2_3Q"].max()
            df_sub["diff_baseline"] = np.where(
                r2_df["r2_2T"] < 0, np.nan, df_sub["diff_baseline"]
            )
        null_diffs.append(max_diff)
    return df_sub, null_diffs


baseline_parent_switch, null_parent_switch = get_null_dist_switch(parent_r2_df_switch)
baseline_self_switch, null_self_switch = get_null_dist_switch(self_r2_df_switch)

In [ ]:
sig_parent_switch, non_sig_parent_switch = get_p_vals(
    baseline_parent_switch,
    null_parent_switch,
    diff_col="diff_baseline",
    title="Null (via switching) and observed distribution for PARENT 3Q > 2T hypothesis",
    show_plot=True,
    xlim=(-0.25, 0.4),
    ylim=(0, 150),
)
sig_self_switch, non_sig_self_switch = get_p_vals(
    baseline_self_switch,
    null_self_switch,
    diff_col="diff_baseline",
    title="Null (via switching) and observed distribution for SELF 3Q > 2T hypothesis",
    show_plot=True,
    xlim=(-0.15, 0.3),
    ylim=(0, 150),
)

# Part 2

- Look at effect size in sig models
- TODO: Subset baseline dfs to only contain sig model configs

In [ ]:
## AVG ABSOLUTE ERROR
def get_cohens_t_df(df, num_layers, num_repeats=10):
    df_layered = df[df["num_layers"] == num_layers].copy()
    reduced_col = "pred_stage1" if num_layers == 2 else "pred_stage2_total"

    # Diff in abs error
    reduced_error = abs(df_layered["true"] - df_layered[reduced_col])
    full_error = abs(df_layered["true"] - df_layered["pred_stage3_total"])
    df_layered["diff_abs_error"] = reduced_error - full_error
    # Avg + stdev of diff for each model
    result = (
        df_layered.groupby(
            ["num_layers", "agg_type", "llama_model", "desc_seg", "alpha"]
        )["diff_abs_error"]
        .agg(avg_diff="mean", std_diff="std", n="count")
        .reset_index()
    )

    result["n"] = result["n"] / num_repeats  # 10 repeats
    assert result["n"].nunique() == 1, f"Unique: {result["n"].unique()}"
    result["cohens_d"] = result["avg_diff"] / result["std_diff"]
    result["t_stat"] = result["cohens_d"] * np.sqrt(result["n"])
    # result["cohens_d"]
    return result

In [ ]:
baseline_dir = BASE_PATH / "perm_res/preds/baseline"
# 71 total
parent_baseline_df = pd.read_parquet(baseline_dir / "parent/pred_df_combined.parquet")
# 103 total
self_baseline_df = pd.read_parquet(baseline_dir / "self/pred_df_combined.parquet")

Cohens D

- avg(diff) / stdev(diff) for each model

In [ ]:
cohen_parent_2 = get_cohens_t_df(parent_baseline_df, 2)
cohen_parent_3 = get_cohens_t_df(parent_baseline_df, 3)
cohen_self_2 = get_cohens_t_df(self_baseline_df, 2)
cohen_self_3 = get_cohens_t_df(self_baseline_df, 3)

In [ ]:
cohen_parent_2[
    (cohen_parent_2["agg_type"] == "proba-limited")
    & (cohen_parent_2["llama_model"] == "Llama-3.3-8B-Instruct")
    & (cohen_parent_2["desc_seg"] == "original")
    & (cohen_parent_2["alpha"] == "1.0")
]

In [ ]:
cohen_parent_2.sort_values("avg_diff", ascending=False)

In [ ]:
cohen_self_3.sort_values("cohens_d", ascending=False)

## P-vals w/ T-stat

In [ ]:
def collapse_hypothesis_w_t(df1, df2):
    """
    Collapses hypothesis across 2 and 3 layers by taking the min of the baseline diffs.
    Returns a df with columns: agg, llama_model, desc_seg, alpha, min_diff
    """
    combined = pd.merge(
        df1,
        df2,
        on=["agg_type", "llama_model", "desc_seg", "alpha"],
        suffixes=("_2", "_3"),
    )

    combined["min_t_stat"] = combined.apply(
        lambda row: min(row.t_stat_2, row.t_stat_3), axis=1
    )
    return combined.sort_values("min_t_stat", ascending=False)


def get_collapsed_null_t(
    impact_type, perm_dir_base=BASE_PATH / "perm_res" / "preds", single_repeat=False
):
    """
    Creates null distribution by first collapsing hypothesis axis (min) then collapsing model axis (max)
    for each permutation. Results in a single diff value per permutation to create a shared null distribution
    for a given impact_type.

    Params
    -----
    impact_type: str
        One of ['self', 'parent']
    """
    null_dist = []
    baseline_df = None
    for perm_dir in perm_dir_base.iterdir():
        pred_df = pd.read_parquet(perm_dir / impact_type / "pred_df_combined.parquet")
        if single_repeat:
            pred_df = pred_df[pred_df["repeat"] == 0]
        # Add cohen_d + t_stat
        num_repeats = 1 if single_repeat else 10
        cohen_t_2 = get_cohens_t_df(pred_df, 2, num_repeats=num_repeats)
        cohen_t_3 = get_cohens_t_df(pred_df, 3, num_repeats=num_repeats)
        # Duplicate 2 layer to match 3 layer alphas
        cohen_t_2_dup = expand_alpha_rows(cohen_t_2)
        # Collpase hypothesis axis
        # t stat of 2 layer df is for hypotheis 2T > 2S
        # t stat of 3 layer df is for hypotheis 3T > 3Q
        combined = collapse_hypothesis_w_t(cohen_t_2_dup, cohen_t_3)
        if perm_dir.name == "baseline":
            baseline_df = combined
        max_t = max(combined["min_t_stat"])
        null_dist.append(max_t)
    return null_dist, baseline_df

### Global

In [ ]:
# cohen_parent_2_dup = expand_alpha_rows(cohen_parent_2)
# cohen_self_2_dup = expand_alpha_rows(cohen_self_2)
# parent_combined = collapse_hypothesis_w_t(cohen_parent_2_dup, cohen_parent_3)
# self_combined = collapse_hypothesis_w_t(cohen_self_2_dup, cohen_self_3)

In [ ]:
null_dist_parent, parent_baseline = get_collapsed_null_t(impact_type="parent")

In [ ]:
null_dist_self, self_baseline = get_collapsed_null_t(impact_type="self")

p-vals

In [ ]:
parent_baseline["p_val"] = parent_baseline["min_t_stat"].apply(
    lambda baseline: get_ranking(baseline, null_dist_parent)
)
parent_combined_sig = parent_baseline[parent_baseline["p_val"] < 0.05].sort_values(
    "min_t_stat", ascending=False
)
print(f"Significant model configs: {len(parent_combined_sig)} / {len(parent_baseline)}")
parent_combined_sig

In [ ]:
parent_combined_sig.sort_values("cohens_d_3", ascending=False)

In [ ]:
self_baseline["p_val"] = self_baseline["min_t_stat"].apply(
    lambda baseline: get_ranking(baseline, null_dist_self)
)
self_combined_sig = self_baseline[self_baseline["p_val"] < 0.05].sort_values(
    "min_t_stat", ascending=False
)
print(f"Significant model configs: {len(self_combined_sig)} / {len(self_baseline)}")
self_combined_sig

### With single repeat

In [ ]:
null_dist_parent_single, parent_baseline_single = get_collapsed_null_t(
    impact_type="parent", single_repeat=True
)
null_dist_self_single, self_baseline_single = get_collapsed_null_t(
    impact_type="self", single_repeat=True
)

In [ ]:
parent_baseline_single["p_val"] = parent_baseline_single["min_t_stat"].apply(
    lambda baseline: get_ranking(baseline, null_dist_parent_single)
)
parent_combined_sig_single = parent_baseline_single[
    parent_baseline_single["p_val"] < 0.05
].sort_values("min_t_stat", ascending=False)
print(
    f"Significant model configs: {len(parent_combined_sig_single)} / {len(parent_baseline_single)}"
)
parent_combined_sig_single

In [ ]:
self_baseline_single["p_val"] = self_baseline_single["min_t_stat"].apply(
    lambda baseline: get_ranking(baseline, null_dist_self_single)
)
self_combined_sig_single = self_baseline_single[
    self_baseline_single["p_val"] < 0.3
].sort_values("min_t_stat", ascending=False)
print(
    f"Significant model configs: {len(self_combined_sig_single)} / {len(self_baseline_single)}"
)
self_combined_sig_single